## 0. Configuration

In [ ]:
pip install fastparquet

In [ ]:
pip install py3dep

In [ ]:
import json
import re
import warnings
import zipfile
from itertools import product
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=UserWarning)
rng = np.random.default_rng(11)

CFG = dict(
    crs_m="EPSG:3310",                       # California Albers, metres
    scales_m=(30, 90, 300, 1000),            # terrain stencil radii
    n_folds=5,
    alpha=0.10,                              # conformal miscoverage (90% PI)
    imts=("PGA", "SA(1.0)"),
    # HayWired M7.0 rupture: ~83 km of the Hayward fault from San Pablo Bay to Fremont
    # (Detweiler & Hudnut 2017). Surface trace as (lon, lat) vertices NW -> SE, read off
    # the Quaternary Fault database; replaces the Qfaults download and linemerge step,
    # which collapsed to a 9 km fragment because the database segments do not snap.
    hayward_trace=[(-122.395, 38.020), (-122.300, 37.905), (-122.220, 37.820),
                   (-122.120, 37.710), (-122.020, 37.600), (-121.930, 37.490)],
    rupture=dict(mag=7.0, rake=180.0, dip=90.0, usd=0.0, lsd=13.0, spacing=1.0),
    near_km=150.0,                           # facilities reported as "in the footprint"
    bay_wkt="POLYGON((-122.9 37.2,-121.5 37.2,-121.5 38.4,-122.9 38.4,-122.9 37.2))",
    urls=dict(
        vs30="https://www.sciencebase.gov/catalog/file/get/5f44290282ce4c3d1222da63?f=__disk__01%2Ff6%2F97%2F01f6979edcca8e11d4df3eb5b08b6b11a43d3bb8",
        vs30map="https://www.sciencebase.gov/catalog/file/get/5a5fa029e4b06e28e9bfc43a?f=__disk__2b%2F15%2Fc5%2F2b15c57ae4bc092da9b3d8af7a2a32d8304f8a30",
        cms="https://data.cms.gov/provider-data/sites/default/files/resources/0970c657767e40f8521187d02ba8d0ed_1792181744/NH_ProviderInfo_Oct2026.csv",
        svi="https://svi.cdc.gov/Documents/Data/2022/csv/states/California.csv",
        tracts="https://www2.census.gov/geo/tiger/TIGER2023/TRACT/tl_2023_06_tract.zip",
        sgmc="https://www.sciencebase.gov/catalog/file/get/5888bf4fe4b05ccb964bab9d?f=__disk__24%2Ff6%2Fe1%2F24f6e139c181fd9fe43df2aaf7f50b1c5b3b6297",
    ),
)

DATA, OUT = Path("data"), Path("outputs")
for p in (DATA, OUT):
    p.mkdir(parents=True, exist_ok=True)


def fetch(key, name):
    p = DATA / name
    if not p.exists():
        print(f"downloading {key} -> {p}")
        urlretrieve(CFG["urls"][key], p)
    return p


def cached(name, build):
    p = DATA / f"{name}.parquet"
    if p.exists():
        return pd.read_parquet(p)
    df = build()
    df.to_parquet(p)
    return df


def unzip(key, name, dest):
    d = DATA / dest
    if not d.exists():
        zipfile.ZipFile(fetch(key, name)).extractall(d)
    return d


def sample_raster(path, lon, lat):
    """Sample a raster at lon/lat (EPSG:4326); nodata, zeros and negatives -> NaN."""
    import rasterio
    with rasterio.open(path) as src:
        vals = np.array([v[0] for v in src.sample(zip(lon, lat))], float)
        if src.nodata is not None:
            vals[vals == src.nodata] = np.nan
    vals[~np.isfinite(vals) | (vals <= 0)] = np.nan
    return vals


def to_points(df):
    return gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df["lon"], df["lat"]), crs=4326)


def col(df, *needles, required=True):
    """First column whose upper-cased name contains every needle."""
    for c in df.columns:
        if all(n.upper() in c.upper() for n in needles):
            return c
    if required:
        raise KeyError(f"no column matching {needles}; have {list(df.columns)}")
    return None


## 1. Labels, facilities, background points

Measured Vs30 (Yong et al. compilation, 2020 update), California only, modelled as log Vs30.
Every entry is a direct field measurement, so no method filter; `method` is kept so ReMi
(the largest group) can be tested as a feature or weight later. California membership is a
spatial join against the tract polygons because the compilation carries no state field.

CMS ships coordinates and a geocoding footnote, so facilities need a quality filter, not
geocoding. Background points are uniform over California and are the control group for the
siting test in Section 12.

In [ ]:
raw = pd.read_csv(DATA / "cms_nh.csv", low_memory=False, nrows=5000)
raw.columns = raw.columns.str.lower().str.replace(r"\W+", "_", regex=True).str.strip("_")
ca = raw[raw["state"].astype(str).str.strip().eq("CA")]
print(ca[["latitude", "longitude"]].dtypes.to_dict())
print(ca[["latitude", "longitude"]].head(5).to_string())
print("non-null:", ca[["latitude", "longitude"]].notna().mean().round(3).to_dict())
print([c for c in raw.columns if any(k in c for k in ("locat", "geo", "lat", "lon", "address", "zip"))])

In [ ]:
TRACTS = gpd.read_file(next((unzip("tracts", "tracts_ca.zip", "tracts")).rglob("*.shp"))).to_crs(4326)
CA_POLY = TRACTS.union_all()

def build_labels():
    df = pd.read_csv(fetch("vs30", "vs30_compilation_2020.csv"), encoding="utf-8-sig")
    df = df.rename(columns={col(df, "ID"): "site_id", col(df, "LAT"): "lat", col(df, "LON"): "lon",
                            col(df, "VS30"): "vs30", col(df, "METHOD"): "method"})
    df["vs30"] = pd.to_numeric(df["vs30"], errors="coerce")
    df = df[df["vs30"].between(80, 2500)].dropna(subset=["lon", "lat"])
    df = df[to_points(df).within(CA_POLY).to_numpy()]
    df["ln_vs30"] = np.log(df["vs30"])
    return (df.groupby("site_id")
              .agg(lon=("lon", "mean"), lat=("lat", "mean"), ln_vs30=("ln_vs30", "mean"),
                   method=("method", "first"), n_meas=("vs30", "size"))
              .reset_index())


FOOTNOTE_OK = {"", "nan", "none"}


def build_facilities():
    df = pd.read_csv(fetch("cms", "cms_nh.csv"), low_memory=False)
    df.columns = df.columns.str.lower().str.replace(r"\W+", "_", regex=True).str.strip("_")
    df = df[df[col(df, "state")].astype(str).str.strip().eq("CA")]

    c_id = (col(df, "ccn", required=False) or col(df, "provider_number", required=False)
            or col(df, "provider_id"))
    names = {c_id: "provider_id",
             col(df, "latitude"): "lat", col(df, "longitude"): "lon",
             col(df, "certified_beds"): "beds",
             col(df, "overall_rating"): "rating",
             col(df, "ownership_type"): "ownership"}
    print("CMS columns resolved:", names)
    df = df.rename(columns=names)
    for c in ("lat", "lon"):
        df[c] = pd.to_numeric(df[c], errors="coerce")

    c_fn = col(df, "geocoding_footnote", required=False)
    footnote = (df[c_fn].astype("string").fillna("").str.strip().str.lower()
                if c_fn else pd.Series("", index=df.index, dtype="string"))
    keep = footnote.isin(FOOTNOTE_OK) & df[["lat", "lon"]].notna().all(axis=1)
    print(f"facilities: dropped {(~keep).sum()} of {len(df)} on coordinate quality "
          f"(footnotes present: {footnote[~footnote.isin(FOOTNOTE_OK)].value_counts().to_dict()})")
    if keep.sum() == 0:
        raise RuntimeError("no facilities survived; check the resolved columns above")

    out = df.loc[keep, ["provider_id", "lat", "lon", "beds", "rating", "ownership"]].reset_index(drop=True)
    svi = pd.read_csv(fetch("svi", "svi_ca.csv"), dtype={"FIPS": str})
    j = gpd.sjoin(to_points(out), TRACTS[["GEOID", "geometry"]], how="left", predicate="within")
    j = j[~j.index.duplicated()]
    out["svi"] = j["GEOID"].map(svi.set_index("FIPS")["RPL_THEMES"]).replace(-999, np.nan).to_numpy()
    return out


def sample_background(n):
    minx, miny, maxx, maxy = CA_POLY.bounds
    pts = gpd.GeoSeries(gpd.points_from_xy(
        rng.uniform(minx, maxx, 4 * n), rng.uniform(miny, maxy, 4 * n)), crs=4326)
    keep = pts[pts.within(CA_POLY)].iloc[:n]
    return pd.DataFrame({"lon": keep.geometry.x.values, "lat": keep.geometry.y.values})


labels = cached("labels", build_labels)
fac = cached("facilities", build_facilities)
bg = cached("background", lambda: sample_background(len(fac)))
print(labels.shape, labels["method"].value_counts().head(6).to_dict())
print(fac.shape, f"svi missing {fac['svi'].isna().sum()}")
print(bg.shape)

## 2. Terrain features (needs the 3DEP service)

Elevation is sampled on a nine-point stencil at four radii instead of rasterising the
state. Multiscale slope is the testable departure from the single-resolution slope proxy.
This cell computes and caches terrain for labels, facilities and background points in one
go so the rest of the notebook can run without network access.

In [ ]:
import py3dep

OFF = np.array([(0, 0), (1, 0), (-1, 0), (0, 1), (0, -1),
                (1, 1), (1, -1), (-1, 1), (-1, -1)], float)

def stencil_coords(lon, lat, r_m):
    dlat = r_m / 111320.0
    dlon = dlat / np.cos(np.radians(lat))
    return (lon[:, None] + OFF[:, 0] * dlon[:, None],
            lat[:, None] + OFF[:, 1] * dlat)

def terrain_at(lon, lat, r_m):
    gx, gy = stencil_coords(lon, lat, r_m)
    z = np.asarray(py3dep.elevation_bycoords(
        list(zip(gx.ravel(), gy.ravel())), crs=4326, source="tep"), float)
    z = z.reshape(gx.shape)
    dzdx = (z[:, 1] - z[:, 2]) / (2 * r_m)
    dzdy = (z[:, 3] - z[:, 4]) / (2 * r_m)
    lap = (z[:, 1] + z[:, 2] + z[:, 3] + z[:, 4] - 4 * z[:, 0]) / r_m ** 2
    return pd.DataFrame({
        f"slope_{r_m}": np.degrees(np.arctan(np.hypot(dzdx, dzdy))),
        f"curv_{r_m}": lap,
        f"tpi_{r_m}": z[:, 0] - z[:, 1:].mean(1),
        f"rough_{r_m}": z[:, 1:].std(1),
        "elev": z[:, 0],
    })

def terrain_table(df):
    parts = [terrain_at(df["lon"].to_numpy(), df["lat"].to_numpy(), r) for r in CFG["scales_m"]]
    out = pd.concat([p.drop(columns="elev") for p in parts], axis=1)
    out["elev"] = parts[0]["elev"]
    return out.set_index(df.index)


terrain = cached("terrain_labels", lambda: terrain_table(labels))
terrain_fac = cached("terrain_fac", lambda: terrain_table(fac))
terrain_bg = cached("terrain_bg", lambda: terrain_table(bg))
print(terrain.shape, terrain_fac.shape, terrain_bg.shape)
print(f"missing elevation: {terrain['elev'].isna().mean():.3f}")

## 3. Geology and distance to the state boundary

Lithology and age enter as native categoricals with rare classes pooled. The SGMC source
can be the full geodatabase (`sgmc_geodatabase.zip`, layer `SGMC_Geology`) or the
California shapefile from the same ScienceBase item (`sgmc_ca.zip`); columns are resolved
by name fragment because the two products truncate differently. NHD flowlines were not
retrieved, so the only distance feature is distance to the state's outer boundary (coast
plus land border), which is a coarse coast proxy and should be read as such.

In [ ]:
def pool_rare(s, min_n=30):
    keep = s.value_counts()[lambda v: v >= min_n].index
    return s.where(s.isin(keep), "other").astype("category")

import pyogrio

def load_sgmc():
    if (DATA / "sgmc_ca.zip").exists():
        d = unzip("sgmc", "sgmc_ca.zip", "sgmc_ca")
        src = next(p for p in d.rglob("*.shp") if "poly" in p.name.lower() or "geol" in p.name.lower())
        layer = None
    else:
        src = next((unzip("sgmc", "sgmc_geodatabase.zip", "sgmc")).rglob("*.gdb"))
        layer = "SGMC_Geology"
    layer_crs = pyogrio.read_info(src, layer=layer)["crs"]
    mask = gpd.GeoSeries([CA_POLY], crs=4326).to_crs(layer_crs).iloc[0]
    g = gpd.read_file(src, layer=layer, mask=mask).to_crs(4326)
    print(f"SGMC: {len(g)} polygons in the California mask (layer crs {layer_crs})")
    lith = col(g, "GENERALIZ")
    age = col(g, "MIN_MAX_AGE", required=False) or col(g, "AGE")
    return g[[lith, age, "geometry"]].rename(columns={lith: "lithology", age: "age"})

SGMC = load_sgmc()
CONTACTS = gpd.GeoDataFrame(geometry=SGMC.to_crs(CFG["crs_m"]).boundary, crs=CFG["crs_m"])


def geo_table(df):
    pts = to_points(df)
    g = gpd.sjoin(pts, SGMC, how="left", predicate="within").drop(columns="index_right")
    g = g[~g.index.duplicated()]
    near = gpd.sjoin_nearest(pts.to_crs(CFG["crs_m"])[["geometry"]], CONTACTS,
                             how="left", distance_col="d_contact")
    near = near[~near.index.duplicated()]
    return pd.DataFrame({
        "lithology": g["lithology"].fillna("unmapped").astype(str),
        "age": g["age"].fillna("unmapped").astype(str),
        "d_contact": near["d_contact"].to_numpy(),
    }, index=df.index)


geo = cached("geo2_labels", lambda: geo_table(labels))
geo_fac = cached("geo2_fac", lambda: geo_table(fac))
geo_bg = cached("geo2_bg", lambda: geo_table(bg))

CAT = ["lithology", "age"]
for c in CAT:                                   # pool on the labels, align the others
    geo[c] = pool_rare(geo[c])
    for other in (geo_fac, geo_bg):
        other[c] = pd.Categorical(other[c].where(other[c].isin(geo[c].cat.categories), "other"),
                                  categories=geo[c].cat.categories)

X = pd.concat([terrain, geo], axis=1)
Xf = pd.concat([terrain_fac, geo_fac], axis=1)
Xb = pd.concat([terrain_bg, geo_bg], axis=1)
y = labels["ln_vs30"].to_numpy()
NUM = [c for c in X.columns if c not in CAT]
print(X.shape, X.dtypes.value_counts().to_dict())
print("lithology classes:", list(geo["lithology"].cat.categories))
print(f"d_contact (km): median {np.nanmedian(geo['d_contact'])/1000:.2f}, "
      f"max {np.nanmax(geo['d_contact'])/1000:.1f}")

## 4. Spatial folds

Block width is set from the empirical variogram range of the residuals after removing the
lithology mean, not by convention. Folds are assigned by block so held-out sites are
separated from training sites by at least one block width. The Bay Area is withheld
entirely as the regional test and is where the scenario is evaluated.

In [ ]:
def variogram_range(xy, resid, n_bins=25, max_lag_km=120):
    i, j = np.triu_indices(len(xy), 1)
    sub = rng.choice(len(i), size=min(400_000, len(i)), replace=False)
    i, j = i[sub], j[sub]
    h = np.linalg.norm(xy[i] - xy[j], axis=1) / 1000.0
    g = 0.5 * (resid[i] - resid[j]) ** 2
    m = h < max_lag_km
    edges = np.linspace(0, max_lag_km, n_bins + 1)
    b = np.digitize(h[m], edges) - 1
    gamma = np.bincount(b, g[m], n_bins) / np.maximum(np.bincount(b, minlength=n_bins), 1)
    sill = gamma[-5:].mean()
    return float(edges[1:][np.argmax(gamma >= 0.95 * sill)]), edges[1:], gamma


pts_m = to_points(labels).to_crs(CFG["crs_m"])
xy = np.c_[pts_m.geometry.x, pts_m.geometry.y]
resid = y - pd.Series(y).groupby(geo["lithology"].values).transform("mean").to_numpy()
vrange_km, vg_edges, vg_gamma = variogram_range(xy, resid)

block_m = max(vrange_km, 25.0) * 1000
block_id = np.unique((xy // block_m).astype(int), axis=0, return_inverse=True)[1].ravel()
fold = pd.Series(rng.permutation(block_id.max() + 1) % CFG["n_folds"])[block_id].to_numpy()

BAY = gpd.GeoSeries.from_wkt([CFG["bay_wkt"]], crs=4326).iloc[0]
is_bay = to_points(labels).within(BAY).to_numpy()
print(f"range {vrange_km:.0f} km | block {block_m/1000:.0f} km | bay holdout {is_bay.sum()} of {len(y)}")
print("sites per fold (non-bay):", np.bincount(fold[~is_bay]))

## 5. Baselines

Two anchors. The slope-only baseline is an isotonic fit of log Vs30 on 30 m slope, fit on
training folds only, a calibrated stand-in for the Wald & Allen (2007) lookup table. The
operational baseline is the Wills et al. (2015) hybrid California Vs30 map (Thompson,
v2.0), sampled at each site together with its kriging standard-deviation grid. The map
ingested measured sites, so its error at those sites is optimistic; the metadata says
kriging was constrained not to propagate values across geologic units, which narrows but
does not remove that concern.

In [ ]:
%pip install scikit-learn lightgbm shap

In [ ]:
from sklearn.isotonic import IsotonicRegression

VS30MAP = unzip("vs30map", "version_1.0.zip", "vs30map")
for z in VS30MAP.rglob("*.zip"):
    try:
        zipfile.ZipFile(z).extractall(VS30MAP / "grids")
    except zipfile.BadZipFile:
        print(f"skipping {z.name}: not a zip ({z.stat().st_size} bytes)")
print(sorted(p.relative_to(VS30MAP).as_posix() for p in VS30MAP.rglob("*") if p.is_file())[:40])

tifs = list(VS30MAP.rglob("*.tif"))
SD_TIF = next(p for p in tifs if p.stem.lower().endswith("_sd"))
VS30_TIF = next(p for p in tifs if p != SD_TIF and "7p5c" in p.name)
print(VS30_TIF.name, "|", SD_TIF.name)

## 6. Model and spatially blocked cross-validation

Tuning happens inside the fold loop so reported error is not contaminated by
hyperparameter selection. ReMi dominates the labels; `method` is held out of the features
so the model is usable where no measurement exists, and a method sensitivity check is
left to the appendix.

In [ ]:
def slope_baseline(tr, te):
    """Isotonic fit of ln Vs30 on 30 m slope, trained on tr only, predicted at te."""
    s = X[f"slope_{CFG['scales_m'][0]}"].to_numpy()
    m = np.isfinite(s[tr]) & np.isfinite(y[tr])
    iso = IsotonicRegression(increasing=True, out_of_bounds="clip").fit(s[tr][m], y[tr][m])
    out = np.full(len(te), np.nan)
    ok = np.isfinite(s[te])
    out[ok] = iso.predict(s[te][ok])
    return out


operational = np.log(sample_raster(VS30_TIF, labels["lon"], labels["lat"]))
operational_sd = sample_raster(SD_TIF, labels["lon"], labels["lat"])
m = np.isfinite(operational)
print(f"operational map covers {m.mean():.3f} of sites | "
      f"rmse at measured sites {np.sqrt(np.mean((y[m] - operational[m]) ** 2)):.3f} "
      f"(optimistic: map ingested these sites) | median SD {np.nanmedian(operational_sd):.3f}")

In [ ]:
import lightgbm as lgb

PARAM_GRID = [dict(num_leaves=n, learning_rate=lr, min_child_samples=m, n_estimators=600)
              for n, lr, m in product((15, 31, 63), (0.03, 0.06), (10, 25))]


def fit_lgb(params, tr, obj="regression", alpha=None):
    return lgb.LGBMRegressor(objective=obj, alpha=alpha, verbose=-1,
                             random_state=11, **params).fit(
        X.iloc[tr], y[tr], categorical_feature=CAT)


def rmse(a, b):
    m = np.isfinite(a) & np.isfinite(b)
    return float(np.sqrt(np.mean((a[m] - b[m]) ** 2)))


def tune(tr):
    inner = fold[tr]
    scores = [np.mean([rmse(y[tr][inner == k],
                            fit_lgb(p, tr[inner != k]).predict(X.iloc[tr[inner == k]]))
                       for k in np.unique(inner)])
              for p in PARAM_GRID]
    return PARAM_GRID[int(np.argmin(scores))]


rows, best_by_fold = [], {}
preds = np.full(len(y), np.nan)
slope_oof = np.full(len(y), np.nan)
for k in range(CFG["n_folds"]):
    tr = np.where((fold != k) & ~is_bay)[0]
    te = np.where((fold == k) & ~is_bay)[0]
    best_by_fold[k] = best = tune(tr)
    preds[te] = fit_lgb(best, tr).predict(X.iloc[te])
    slope_oof[te] = slope_baseline(tr, te)
    rows.append(dict(fold=k, n=len(te),
                     gbm=rmse(y[te], preds[te]),
                     slope=rmse(y[te], slope_oof[te]),
                     operational=rmse(y[te], operational[te])))
    print(f"fold {k}: {rows[-1]}")

cv = pd.DataFrame(rows)
print(cv.drop(columns=["fold", "n"]).agg(["mean", "std"]).round(3))

In [ ]:
# Does the multiscale stencil add anything over a single-resolution terrain set?
TERRAIN_PFX = ("slope_", "curv_", "tpi_", "rough_")
r0 = CFG["scales_m"][0]
sets = {
    "full": list(X.columns),
    f"terrain@{r0}m only": [c for c in X.columns
                            if not c.startswith(TERRAIN_PFX) or c.endswith(f"_{r0}")],
    "no terrain (elev+geo)": [c for c in X.columns if not c.startswith(TERRAIN_PFX)],
    "terrain only (no geo, no elev)": [c for c in X.columns
                                       if c.startswith(TERRAIN_PFX)],
}

abl = {}
for name, cols in sets.items():
    oof = np.full(len(y), np.nan)
    cats = [c for c in CAT if c in cols]
    for k in range(CFG["n_folds"]):
        tr = np.where((fold != k) & ~is_bay)[0]
        te = np.where((fold == k) & ~is_bay)[0]
        mdl = lgb.LGBMRegressor(objective="regression", verbose=-1, random_state=11,
                                **best_by_fold[k]).fit(X.iloc[tr][cols], y[tr],
                                                        categorical_feature=cats)
        oof[te] = mdl.predict(X.iloc[te][cols])
    abl[name] = dict(n_features=len(cols),
                     rmse=rmse(y[~is_bay], oof[~is_bay]),
                     **{f"fold{k}": rmse(y[(fold == k) & ~is_bay], oof[(fold == k) & ~is_bay])
                        for k in range(CFG["n_folds"])})
abl = pd.DataFrame(abl).T
print(abl.round(3))
abl.to_csv(OUT / "table2c_ablation.csv")

## 7. Calibrated intervals, compared with the published map uncertainty

Conformalized quantile regression. Coverage guarantees assume exchangeability, which
spatial blocking deliberately breaks, so the reported evidence is empirical coverage on
the withheld Bay Area region. The same test is applied to the operational map's own
interval, mean ± z·SD, so the comparison is calibration against calibration at the same
nominal level.

In [ ]:
from scipy.stats import norm

a = CFG["alpha"]
tr_all = np.where(~is_bay)[0]
cal = tr_all[fold[tr_all] == 0]
fit = tr_all[fold[tr_all] != 0]

best = tune(fit)
lo_m = fit_lgb(best, fit, "quantile", a / 2)
hi_m = fit_lgb(best, fit, "quantile", 1 - a / 2)

score = np.maximum(lo_m.predict(X.iloc[cal]) - y[cal], y[cal] - hi_m.predict(X.iloc[cal]))
q = np.quantile(score, min(1.0, np.ceil((len(cal) + 1) * (1 - a)) / len(cal)))


def interval(Xi):
    return lo_m.predict(Xi) - q, hi_m.predict(Xi) + q


def coverage(lo, hi, yy):
    m = np.isfinite(lo) & np.isfinite(hi) & np.isfinite(yy)
    return float(np.mean((yy[m] >= lo[m]) & (yy[m] <= hi[m]))), float(np.mean(hi[m] - lo[m]))


bay_idx = np.where(is_bay)[0]
blo, bhi = interval(X.iloc[bay_idx])
z = norm.ppf(1 - a / 2)
olo, ohi = operational[bay_idx] - z * operational_sd[bay_idx], operational[bay_idx] + z * operational_sd[bay_idx]

calib = pd.DataFrame({
    "conformal GBM": coverage(blo, bhi, y[bay_idx]),
    "operational ± z·SD": coverage(olo, ohi, y[bay_idx]),
}, index=["coverage", "mean width (ln)"]).T
calib["nominal"] = 1 - a
print(calib.round(3))

# Regional test: point model trained on every non-Bay label, scored on the 403 Bay sites.
bay_model = fit_lgb(best, tr_all)
bay_rmse = pd.Series(dict(
    gbm=rmse(y[bay_idx], bay_model.predict(X.iloc[bay_idx])),
    slope=rmse(y[bay_idx], slope_baseline(tr_all, bay_idx)),
    operational=rmse(y[bay_idx], operational[bay_idx]),
), name="bay_rmse")
print(bay_rmse.round(3))

model = fit_lgb(best, np.arange(len(y)))      # final point model, all labels incl. Bay


## 8. Attribution and extrapolation

SHAP explains predictions; it does not detect extrapolation, so novelty is measured
separately as distance to the nearest training label in scaled feature space.

In [ ]:
import shap
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

shap_values = shap.TreeExplainer(model).shap_values(X)

scaler = StandardScaler().fit(X.loc[:, NUM].fillna(X.loc[:, NUM].median()))
nn = NearestNeighbors(n_neighbors=1).fit(scaler.transform(X.loc[:, NUM].fillna(X.loc[:, NUM].median())))


def novelty(df):
    return nn.kneighbors(scaler.transform(df.loc[:, NUM].fillna(X.loc[:, NUM].median())))[0].ravel()


print(pd.Series(np.abs(shap_values).mean(0), index=X.columns)
      .sort_values(ascending=False).head(10).round(3))

## 9. Facilities

Predicted and operational ln Vs30 at each facility, with the conformal interval, feature
novelty, distance to the nearest measured site, and SVI quartile.

In [ ]:
fac["svi_q"] = pd.qcut(fac["svi"], 4, labels=["Q1", "Q2", "Q3", "Q4"])
fac["ln_vs30_op"] = np.log(sample_raster(VS30_TIF, fac["lon"], fac["lat"]))
fac["sd_op"] = sample_raster(SD_TIF, fac["lon"], fac["lat"])
fac["ln_vs30_ml"] = model.predict(Xf)
fac["lo"], fac["hi"] = interval(Xf)
fac["novelty"] = novelty(Xf)

fac_m = to_points(fac).to_crs(CFG["crs_m"])
fac["d_label_km"] = NearestNeighbors(n_neighbors=1).fit(xy).kneighbors(
    np.c_[fac_m.geometry.x, fac_m.geometry.y])[0].ravel() / 1000
print(fac[["ln_vs30_op", "ln_vs30_ml", "novelty", "d_label_km"]].describe().round(2))

## 10. Scenario ground motion

BSSA14 is evaluated twice per facility, changing only Vs30. Source and path terms are
identical and cancel, so the difference is the site effect alone, with the nonlinear term
included natively because both evaluations share a reference-rock motion. The rupture is
a vertical M7.0, 0–13 km, on a six-vertex Hayward trace fixed in `CFG` from the HayWired
geometry (San Pablo Bay to Fremont). The earlier Qfaults `linemerge` produced a 9 km
fragment in the wrong place; the hard-coded trace is checked against the published
rupture length. Facilities with no operational Vs30 value (outside the raster) get no
difference rather than a default-soil one.


In [ ]:
from shapely.geometry import LineString
from openquake.hazardlib.geo import Line, Point, Mesh
from openquake.hazardlib.geo.surface import SimpleFaultSurface
from openquake.hazardlib.gsim.boore_2014 import BooreEtAl2014
from openquake.hazardlib.contexts import simple_cmaker
from openquake.baselib import __version__ as oq_version
assert tuple(int(v) for v in oq_version.split(".")[:2]) >= (3, 19), oq_version

trace_xy = np.array(CFG["hayward_trace"], float)
trace_km = gpd.GeoSeries([LineString(trace_xy)], crs=4326).to_crs(CFG["crs_m"]).length.iloc[0] / 1000
assert 70 < trace_km < 95, f"Hayward trace is {trace_km:.0f} km; HayWired rupture is ~83 km"
print(f"Hayward trace: {len(trace_xy)} vertices, {trace_km:.0f} km")

R = CFG["rupture"]
surface = SimpleFaultSurface.from_fault_data(
    Line([Point(float(x), float(yy)) for x, yy in trace_xy]),
    R["usd"], R["lsd"], R["dip"], R["spacing"])
rjb = surface.get_joyner_boore_distance(Mesh(fac["lon"].to_numpy(), fac["lat"].to_numpy()))
cmaker = simple_cmaker([BooreEtAl2014()], list(CFG["imts"]))


def gmm(ln_vs30):
    """Mean ln ground motion per IMT (n_imt x n_sites) for the given ln Vs30."""
    ctx = cmaker.new_ctx(len(fac))
    ctx["mag"], ctx["rake"] = R["mag"], R["rake"]
    ctx["rjb"] = rjb
    ctx["vs30"] = np.clip(np.exp(np.nan_to_num(ln_vs30, nan=np.log(360.0))), 90, 1500)
    return cmaker.get_mean_stds([ctx])[0][0]


op_ok = np.isfinite(fac["ln_vs30_op"].to_numpy())
mean_op, mean_ml = gmm(fac["ln_vs30_op"].to_numpy()), gmm(fac["ln_vs30_ml"].to_numpy())
mean_lo, mean_hi = gmm(fac["lo"].to_numpy()), gmm(fac["hi"].to_numpy())
for i, imt in enumerate(CFG["imts"]):
    fac[f"d_{imt}"] = np.where(op_ok, mean_ml[i] - mean_op[i], np.nan)
    fac[f"d_{imt}_lo"] = np.where(op_ok, np.minimum(mean_lo[i], mean_hi[i]) - mean_op[i], np.nan)
    fac[f"d_{imt}_hi"] = np.where(op_ok, np.maximum(mean_lo[i], mean_hi[i]) - mean_op[i], np.nan)
fac["rjb"] = rjb
fac["near"] = fac["rjb"] <= CFG["near_km"]
fac["excludes_zero"] = ((fac["d_PGA_lo"] > 0) | (fac["d_PGA_hi"] < 0)).where(op_ok)

print(f"rjb: min {rjb.min():.1f} km, median {np.median(rjb):.0f} km | "
      f"{fac['near'].sum()} facilities within {CFG['near_km']:.0f} km")
for label, m in [("all facilities", op_ok), ("within footprint", op_ok & fac["near"].to_numpy())]:
    print(f"\n{label} (n={m.sum()})")
    print(fac.loc[m, [f"d_{i}" for i in CFG["imts"]]].describe().round(3))
    print(f"interval excludes zero: {fac.loc[m, 'excludes_zero'].mean():.3f}")


## 11. Published HayWired field

Not used. The HayWired ShakeMap grid was not retrievable (empty tile-package stub on
ScienceBase, `escweb.wr.usgs.gov` returning 502), and nothing downstream depends on it;
the comparison is listed as future work in the paper.


## 12. Siting test and stratified summaries

The siting test decides whether the facility framing is doing scientific work: if
facilities are indistinguishable from uniformly sampled background points, the application
is decorative. The stratified summaries describe the coverage of the measurement record,
not the distribution of risk.

In [ ]:
bg["ln_vs30_op"] = np.log(sample_raster(VS30_TIF, bg["lon"], bg["lat"]))
bg["disagree"] = model.predict(Xb) - bg["ln_vs30_op"]
blo_b, bhi_b = interval(Xb)
bg["width"] = bhi_b - blo_b
fac["disagree"] = fac["ln_vs30_ml"] - fac["ln_vs30_op"]
fac["width"] = fac["hi"] - fac["lo"]


def boot_ci(v, n=2000):
    s = rng.choice(v, (n, len(v))).mean(1)
    return np.mean(v), *np.quantile(s, [0.025, 0.975])


siting = pd.DataFrame([
    dict(group=g, metric=m, **dict(zip(["mean", "lo", "hi"], boot_ci(d[m].dropna().to_numpy()))))
    for g, d in [("facilities", fac), ("background", bg)]
    for m in ["disagree", "width"]])

strat = (fac.dropna(subset=["svi_q"])
            .groupby("svi_q", observed=True)
            .apply(lambda d: pd.Series(dict(
                n=len(d), width=d["width"].mean(),
                d_label_km=d["d_label_km"].mean(), d_label_km_median=d["d_label_km"].median(),
                novelty=d["novelty"].mean(), d_pga=d["d_PGA"].mean(),
                excl_zero=d["excludes_zero"].mean())), include_groups=False))

print(siting.round(3).to_string(index=False))
print(strat.round(3))


## 13. Figures and tables

In [ ]:
plt.rcParams.update({"font.size": 8, "figure.dpi": 200, "savefig.bbox": "tight"})

fig1, ax = plt.subplots(1, 2, figsize=(7, 3.2))
TRACTS.boundary.plot(ax=ax[0], lw=0.1, color="0.8")
to_points(labels).plot(ax=ax[0], column=fold, cmap="tab10", markersize=3)
ax[0].set_title("measured Vs30 sites by spatial fold")
ax[1].plot(vg_edges, vg_gamma, "o-", ms=3)
ax[1].axvline(vrange_km, color="crimson", ls="--", label=f"range {vrange_km:.0f} km")
ax[1].set(xlabel="lag (km)", ylabel="semivariance")
ax[1].legend()
fig1.savefig(OUT / "fig1_study_area_and_variogram.png")

fig2, ax = plt.subplots(1, 3, figsize=(7.5, 2.6), sharex=True, sharey=True)
panels = [("slope", slope_oof[~is_bay]), ("operational", operational[~is_bay]),
          ("gradient-boosted", preds[~is_bay])]
for a_, (name, p) in zip(ax, panels):
    a_.scatter(y[~is_bay], p, s=3, alpha=0.4)
    a_.axline((5.5, 5.5), slope=1, color="k", lw=0.6)
    a_.set_title(f"{name}  rmse={rmse(y[~is_bay], p):.3f}")
    a_.set_xlabel("measured ln Vs30")
ax[0].set_ylabel("predicted")
fig2.savefig(OUT / "fig2_predicted_vs_measured.png")

fig3 = plt.figure(figsize=(5, 4))
shap.summary_plot(shap_values, X, max_display=12, show=False)
fig3.savefig(OUT / "fig3_shap_summary.png")

fig4, ax = plt.subplots(1, 2, figsize=(7, 3.4))
TRACTS.boundary.plot(ax=ax[0], lw=0.1, color="0.85")
s = ax[0].scatter(fac["lon"], fac["lat"], c=fac["d_PGA"], s=6, cmap="RdBu_r", vmin=-0.4, vmax=0.4)
ax[0].plot(trace_xy[:, 0], trace_xy[:, 1], color="k", lw=1)
ax[0].set(xlim=(-123.2, -121.2), ylim=(36.9, 38.6), title="Δ ln PGA (ML − operational)")
fig4.colorbar(s, ax=ax[0], shrink=0.8)
nf = fac[fac["near"] & fac["d_PGA"].notna()]
ax[1].errorbar(nf["rjb"], nf["d_PGA"],
               yerr=[np.clip(nf["d_PGA"] - nf["d_PGA_lo"], 0, None),
                     np.clip(nf["d_PGA_hi"] - nf["d_PGA"], 0, None)],
               fmt="o", ms=2, lw=0.4, alpha=0.5)
ax[1].axhline(0, color="k", lw=0.6)
ax[1].set(xlabel="Rjb (km)", ylabel="Δ ln PGA",
          title=f"site-term difference, 90% interval (Rjb ≤ {CFG['near_km']:.0f} km)")
fig4.savefig(OUT / "fig4_facility_map.png")

fig5, ax = plt.subplots(1, 2, figsize=(7, 2.8))
ax[0].hist([fac["disagree"].dropna(), bg["disagree"].dropna()], bins=40, density=True,
           label=["facilities", "background"])
ax[0].set(xlabel="ln Vs30 disagreement (ML − operational)", ylabel="density")
ax[0].legend()
xq = np.arange(len(strat))
b1 = ax[1].bar(xq - 0.2, strat["width"], 0.4, color="C0", label="90% interval width (ln Vs30)")
ax1b = ax[1].twinx()
b2 = ax1b.bar(xq + 0.2, strat["d_label_km_median"], 0.4, color="C1",
              label="median distance to nearest Vs30 site (km)")
ax[1].set(xticks=xq, xticklabels=strat.index, xlabel="SVI quartile", ylabel="interval width")
ax1b.set_ylabel("distance (km)")
ax[1].set_ylim(0, strat["width"].max() * 1.35)
ax1b.set_ylim(0, strat["d_label_km_median"].max() * 1.35)
ax[1].legend(handles=[b1, b2], loc="upper left", fontsize=6.5, frameon=False)
fig5.savefig(OUT / "fig5_siting_and_knowledge.png")

cv.to_csv(OUT / "table2_skill.csv", index=False)
bay_rmse.to_csv(OUT / "table2d_bay_rmse.csv")
calib.to_csv(OUT / "table2b_calibration.csv")
strat.to_csv(OUT / "table3_stratified.csv")
fac.to_parquet(OUT / "facility_results.parquet")
print("wrote", sorted(p.name for p in OUT.iterdir()))


In [ ]:
for name, d in [("fac", fac), ("bg", bg)]:
    print(name, "non-finite ln_vs30_op:", (~np.isfinite(d["ln_vs30_op"])).sum(),
          "| raw ≤ 0:", (np.exp(d["ln_vs30_op"]) <= 0).sum())